# R4 · 0 · Sorotipo do SINAN (opcional)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 a 3):
1. suba **`r4_lib.py`** (pasta `data_imdc_2026/RODADA4/`) e confira que `r3_lib.py` e `r2_lib.py` estão lá;
2. as pastas `resultados_uf_r4`, `revisoes_r4`, `cache_r4` e `avaliacao_r4` são criadas sozinhas.

**Usa a rodada 3:** o normal (ENS_grafos3_λ) vem de `cache_r3/normal_ens_grafos3_bb_clim.parquet` e as misturas da R3 entram na comparação.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r4/uf/`; as revisões de julho a outubro, para `revisoes_r4/uf/`; as tabelas, para `avaliacao_r4/` (com data e hora no nome e uma cópia `_ultimo`).

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (mar, sorotipo por ano, variáveis, detector por método × corte, parâmetros, um arquivo por modelo × temporada, os pesos das redes a cada 5 épocas, um PNG por UF) e o que já existe é pulado.

CPU comum, **30–60 min** na primeira vez (depois é instantâneo). Rode **antes** do R4_1. Se pular, o detector segue sem o sorotipo.

**O que faz:** baixa as notificações de dengue de 2010 a 2026 do OpenDataSUS (`DENGBRaa.csv.zip`, 15–190 MB por ano), lê só 5 colunas (UF de residência, semana dos sintomas, sorotipo e classificação) e conta, por UF × semana, os casos com sorotipo identificado (DENV-1 a DENV-4). Fica um arquivo por ano em `cache_r4/sorotipo/`. O zip é apagado depois de lido.

**Por que:** a troca de sorotipo encontra uma população sem imunidade para ele. Em Singapura, a competição entre sorotipos somada ao clima previu os surtos grandes ([Finch et al. 2025](https://doi.org/10.1038/s41467-025-66411-6)). No Brasil, o DENV-3 voltou em 2023 ([Ferreira et al. 2026](https://doi.org/10.1016/j.lana.2026.101521)).

**Cuidados (sem vazamento):**
- o detector só usa semanas até **8 semanas antes** do corte, imitando o tempo até o resultado do exame entrar no sistema;
- os arquivos são os **finais**, mais completos do que os que existiam na época. O ganho medido aqui é, portanto, um teto otimista;
- só uma fração pequena dos casos tem sorotipo; por isso as frações de cada UF são puxadas para a fração nacional.


In [ ]:
NOTEBOOK = 'r4_sorotipo'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r4', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'r4_lib.py', 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    pass
sys.path.insert(0, PASTA_LOCAL)
PASTA_R4 = os.path.join(PASTA_DRIVE, 'resultados_uf_r4')
PASTA_REV4 = os.path.join(PASTA_DRIVE, 'revisoes_r4')
PASTA_C4 = os.path.join(PASTA_DRIVE, 'cache_r4')
PASTA_AV4 = os.path.join(PASTA_DRIVE, 'avaliacao_r4')
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R4, 'uf'), os.path.join(PASTA_REV4, 'uf'), PASTA_C4, os.path.join(PASTA_C4, 'ckpt'), PASTA_AV4):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, r4_lib as R4
for m in (V, L, CM, U, S, R2, R3, R4):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
R4.PESO_ABS = 0.5     # critério misto = (1 − p)·R × referência + p·WIS absoluto relativo. Suba (ex.: 0.7) para priorizar a altura dos picos


In [ ]:
# @title 2 · Baixar e contar (um arquivo por ano; retoma)
SOR = R4.baixar_sorotipo(range(2010, 2027), PASTA_C4)
SOR['ano'] = SOR.sem_pri // 100
tab = SOR.groupby('ano')[['DENV1', 'DENV2', 'DENV3', 'DENV4']].sum()
print('Casos com sorotipo identificado por ano de início dos sintomas:')
display(tab)
(tab.div(tab.sum(1), axis=0) * 100).round(1)


In [ ]:
# @title 3 · Sorotipo dominante por UF nos últimos anos (conferência)
SOR['uf'] = SOR.uf_code.map(R2.SIGLAS)
d = SOR[SOR.ano >= 2021].groupby(['uf', 'ano'])[['DENV1', 'DENV2', 'DENV3', 'DENV4']].sum()
(d.div(d.sum(1), axis=0) * 100).round(0).unstack('ano').fillna(0).astype(int)
